# Data Validation & Exploratory Analysis

This notebook validates the SQL-built churn training table before feature engineering and modeling. The dataset combines customer demographics, all eight 28-day behavior counts, churn labels, and the synthetic subscription extension.

The behavioral data comes from ChurnSim/SocialNet. The richer subscription layer is a portfolio extension, so subscription relationships are interpreted as predictive associations rather than causal effects.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("../data/churn_training_final.csv")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


In [ ]:
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["observation_date"],
)

df["country"] = df["country"].fillna("Unknown")

df["is_churn"] = (
    df["is_churn"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "t": 1,
        "true": 1,
        "1": 1,
        "f": 0,
        "false": 0,
        "0": 0,
    })
)

print("Shape:", df.shape)
print("Date range:", df["observation_date"].min(), "to", df["observation_date"].max())
print("Unique accounts:", df["account_id"].nunique())
display(df.head())


## Data quality


In [ ]:
duplicate_rows = df.duplicated(
    subset=["account_id", "observation_date"]
).sum()

print("Duplicate account-date rows:", duplicate_rows)
print("Unmapped churn labels:", df["is_churn"].isna().sum())

missing = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing[missing > 0]
    .to_frame("missing_rows")
)


## Target distribution

Churn is a rare event in this dataset, so accuracy is not an appropriate primary model-selection metric. Later notebooks emphasize ROC-AUC, PR-AUC, lift, and churn capture in a fixed retention segment.


In [ ]:
target_summary = (
    df["is_churn"]
    .value_counts()
    .rename_axis("is_churn")
    .to_frame("observations")
)

target_summary["share"] = (
    target_summary["observations"]
    / len(df)
)

display(target_summary)

plt.figure(figsize=(6, 4))
plt.bar(
    ["Retained", "Churned"],
    [
        (df["is_churn"] == 0).sum(),
        (df["is_churn"] == 1).sum(),
    ],
)
plt.ylabel("Observations")
plt.title("Churn Class Distribution")
plt.tight_layout()
plt.show()


## Raw behavioral activity

All eight SocialNet behavior types are retained in the final pipeline. Medians are useful here because activity distributions are strongly right-skewed.


In [ ]:
behavior_features = [
    "post_per_month",
    "newfriend_per_month",
    "like_per_month",
    "adview_per_month",
    "dislike_per_month",
    "unfriend_per_month",
    "message_per_month",
    "reply_per_month",
]

behavior_medians = (
    df.groupby("is_churn")[behavior_features]
    .median()
    .T
)

behavior_medians.columns = [
    "retained_median",
    "churned_median",
]

behavior_medians["difference"] = (
    behavior_medians["churned_median"]
    - behavior_medians["retained_median"]
)

display(behavior_medians)


In [ ]:
activity_total = (
    df[behavior_features]
    .sum(axis=1)
)

plot_df = pd.DataFrame({
    "total_activity": activity_total,
    "is_churn": df["is_churn"],
})

cap = plot_df["total_activity"].quantile(0.99)

retained = plot_df.loc[
    (plot_df["is_churn"] == 0)
    & (plot_df["total_activity"] <= cap),
    "total_activity",
]

churned = plot_df.loc[
    (plot_df["is_churn"] == 1)
    & (plot_df["total_activity"] <= cap),
    "total_activity",
]

plt.figure(figsize=(8, 5))
plt.hist(
    retained,
    bins=40,
    alpha=0.5,
    density=True,
    label="Retained",
)
plt.hist(
    churned,
    bins=40,
    alpha=0.5,
    density=True,
    label="Churned",
)
plt.xlabel("Total 28-Day Activity")
plt.ylabel("Density")
plt.title("Activity Distribution: Retained vs Churned")
plt.legend()
plt.tight_layout()
plt.show()


## Subscription relationships


In [ ]:
plan_churn = (
    df.groupby("current_plan", observed=True)["is_churn"]
    .agg(
        observations="count",
        churners="sum",
        churn_rate="mean",
    )
    .sort_values("churn_rate", ascending=False)
)

billing_churn = (
    df.groupby("bill_period_months", observed=True)["is_churn"]
    .agg(
        observations="count",
        churners="sum",
        churn_rate="mean",
    )
)

print("Churn by plan")
display(plan_churn)

print("Churn by billing commitment")
display(billing_churn)


In [ ]:
movement_summary = pd.concat(
    {
        "recent_upgrade": (
            df.groupby("upgrade_last_90d")["is_churn"]
            .agg(
                observations="count",
                churners="sum",
                churn_rate="mean",
            )
        ),
        "recent_downsell": (
            df.groupby("downsell_last_90d")["is_churn"]
            .agg(
                observations="count",
                churners="sum",
                churn_rate="mean",
            )
        ),
    },
    names=["movement", "flag"],
)

display(movement_summary)


## Customer age and tenure


In [ ]:
customer_summary = (
    df.groupby("is_churn")[
        [
            "customer_age",
            "account_tenure_days",
            "current_mrr",
        ]
    ]
    .agg(["median", "mean"])
    .T
)

display(customer_summary)


## EDA takeaway

The dataset is structurally clean at the account-observation level and strongly imbalanced. Raw activity measures are highly skewed, and negative event counts can be difficult to interpret without an activity denominator. The next notebook therefore creates engagement totals and ratio features before modeling.

The subscription fields are synthetic project features. Their associations with churn should not be interpreted as evidence that a plan change, discount, upgrade, or downsell causes churn or retention.
